In [3]:
import math

def llog(v):
    return -float('inf') if v == 0 else math.log(v)

def log_p(traj, obs, pi, tr, em):
    if len(traj) != len(obs):
        raise ValueError("Length mismatch")
    s = 0.0
    for t in range(len(obs)):
        st = traj[t]
        ob = obs[t]
        if t == 0:
            s += llog(pi[st])
        else:
            prev = traj[t - 1]
            s += llog(tr[prev].get(st, 0))
        s += llog(em[st].get(ob, 0))
    if traj[-1] == 'I' and 'end' in tr['I']:
        s += llog(tr['I']['end'])
    return s

pi = {'E': 1.0, '5': 0.0, 'I': 0.0}
tr = {'E': {'E': 0.9, '5': 0.1}, '5': {'I': 1.0}, 'I': {'I': 0.9, 'end': 0.1}}
em = {'E': {'A': 0.25, 'C': 0.25, 'G': 0.25, 'T': 0.25},
      '5': {'A': 0.05, 'C': 0.0, 'G': 0.95, 'T': 0.0},
      'I': {'A': 0.4, 'C': 0.1, 'G': 0.1, 'T': 0.4}}

traj = "EEEEEEEEEEEEEEEEEE5IIIIIII"
obs = "CTTCATGTGAAAGCAGACGTAAGTCA"
print(f"{log_p(traj, obs, pi, tr, em):.2f}")


-41.22


In [4]:
def viterbi(obs, st, pi, tr, em):
    T = len(obs)
    V = [{} for _ in range(T)]
    B = [{} for _ in range(T)]

    s = obs[0]
    for x in st:
        e = em[x].get(s, 0)
        V[0][x] = llog(pi[x]) + llog(e)
        B[0][x] = None

    for t in range(1, T):
        s = obs[t]
        for x in st:
            e = em[x].get(s, 0)
            if e == 0:
                continue
            best = -float('inf')
            prev_best = None
            for y in st:
                p = tr.get(y, {}).get(x, 0)
                if p > 0:
                    score = V[t - 1].get(y, -float('inf')) + llog(p) + llog(e)
                    if score > best:
                        best = score
                        prev_best = y
            if prev_best is not None:
                V[t][x] = best
                B[t][x] = prev_best

    last = T - 1
    curr = max(V[last], key=V[last].get)
    path = [curr]
    for t in range(last, 0, -1):
        curr = B[t].get(curr)
        if curr is None:
            break
        path.insert(0, curr)

    return ''.join(path)

res = viterbi(obs, ['E', '5', 'I'], pi, tr, em)
print(f"{res}")


EEEEEEEEEEEEEEEEEEEEEEEEEE
